# Logistic Regression

## Prepare the Notebook
> Import the needed libraries, and load the data 

In [1]:
# Import the libraries
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.metrics import accuracy_score, precision_score, recall_score, classification_report, roc_auc_score, roc_curve
import warnings
import sys
# Import the related function of the utils.py file
sys.path.append("D:\Data science project\Data science\Codveda-internship\Level 2")
from Level_2.utils import *
warnings.filterwarnings('ignore')

<>:11: SyntaxWarning: invalid escape sequence '\D'
<>:11: SyntaxWarning: invalid escape sequence '\D'
C:\Users\minaa\AppData\Local\Temp\ipykernel_14752\3878761590.py:11: SyntaxWarning: invalid escape sequence '\D'
  sys.path.append("D:\Data science project\Data science\Codveda-internship\Level 2")


In [2]:
### Load the data
# Load the data
reg_df = pd.read_csv(r"D:\Data science project\Data science\Codveda-internship\Level 2\Task 1\models_data.csv")

# preview the data
print("Real estates data:\n")
reg_df

Real estates data:



,name,has_main_facilities,has_security_features,has_parking_safety,has_building_facilities,has_other_features,HighFloor,MidFloor,has_modern,has_facilities,...,propertySubType_Attached Houses,propertySubType_Office,propertySubType_Others(very rare),propertySubType_Retail,log(sqm),log(price),log(price/sqm),Is log(price/sqm) outlier,Is log(sqm) outlier,Is log(price) outlier
0,Alamain (Latin District),1,1,0,0,1,0,1,1,1,...,0,0,0,0,4.531093,15.701671,11.170578,False,False,False
1,Beachfront Tower - B1,0,0,0,0,1,1,0,0,1,...,0,0,0,0,5.986452,18.006740,12.020288,False,False,True
2,PODIA,0,1,1,0,1,1,0,0,0,...,0,1,0,0,4.532599,16.366829,11.834229,False,False,False
3,Mazarine Apartment,1,1,0,0,0,0,1,0,1,...,0,0,0,0,5.529429,16.369399,10.839970,False,False,False
4,Alamain (Latin District),1,1,0,0,1,0,0,0,0,...,0,0,0,0,5.344342,15.981244,10.636903,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
851,Alamain (Latin District),1,1,0,0,1,0,0,0,0,...,0,0,0,0,5.178802,15.552110,10.373308,False,False,False
852,Jade Park,1,0,1,1,0,0,1,0,1,...,0,0,0,0,6.539586,17.045999,10.506413,False,True,False
853,Mamsha Avenue,1,1,0,0,1,0,1,0,1,...,0,0,0,0,5.141664,15.629847,10.488183,False,False,False
854,Alamain (Latin District),1,1,0,0,1,0,1,1,1,...,0,0,0,0,5.335517,15.596041,10.260524,False,False,False


## Non outliers regression

### Set x, and y

In [3]:
outliers_list = reg_df.columns[reg_df.columns.str.startswith("Is")].tolist() # the outliers bool columns
drop_cols = outliers_list + ["propertyCategory_Commercial","name"] # the columns to drop
x, y = set_xy(reg_df, "propertyCategory_Commercial", False, outliers_list, drop_cols)

print(f"Features shape: {x.shape}\n")
print(f"Target values(first 5 rows):\n{y.head()}")

Features shape: (732, 30)

Target values(first 5 rows):
0    0
2    1
3    0
4    0
5    0
Name: propertyCategory_Commercial, dtype: int64


In [4]:
### Split the data into train and test sets
x_train, x_test, y_train, y_test = split_data(x, y, test_size=0.2)

print(f"X_train shape: {x_train.shape}, y_train shape: {y_train.shape}")
print(f"X_test shape: {x_test.shape}, y_test shape: {y_test.shape}")

X_train shape: (585, 30), y_train shape: (585,)
X_test shape: (147, 30), y_test shape: (147,)


### Perform the original model

> Before that I should see the VIF values, because I think there are collinearity between price/sqm, and price or price/sqm, and sqm 

In [5]:
def vifs_scores(x):
    for col in x.columns:
        vif = variance_inflation_factor(x.values, x.columns.get_loc(col))
        print(f"VIF for {col}: {vif}")
vifs_scores(x_train)

VIF for has_main_facilities: 10.635842612398298


VIF for has_security_features: 64.64676912600058
VIF for has_parking_safety: 3.6607171952264186
VIF for has_building_facilities: 5.6163646183782925
VIF for has_other_features: 12.349506706913498
VIF for HighFloor: 1.8752864195803827
VIF for MidFloor: 2.5811485310301805
VIF for has_modern: 12.185563117769815
VIF for has_facilities: 7.8856992093035
VIF for has_green_view: 8.185853226175643
VIF for has_water_feature: 16.227638657486395
VIF for Bedroom_1: 14.282981282292395
VIF for Bedroom_2: 20.489959228027015
VIF for Bedroom_3: 58.01144063475464
VIF for Bedroom_above 3: 21.623176963592353
VIF for Bathroom_1: 1.7408750593357205
VIF for Bathroom_2: 2.696666512514538
VIF for Bathroom_above 2: 4.508065483816534
VIF for city_Mansoura: 11.529566201751743
VIF for city_New Administrative Capital: 9.762250635370302
VIF for city_New Cairo: 2.053436158752621
VIF for city_North Coast: 16.149712031939753
VIF for city_Old Cairo: 5.6023242796756225
VIF for propertySubType_Attached Houses: 7.10816533908

> I should drop price/sqm, and price to prevent high values, combine has_main_facilities, with building facilities

In [6]:
collinearity_features =  ["log(price/sqm)", "log(price)"] # the perfect collinearity features
x.drop(columns= collinearity_features, axis = 1, inplace=True) # drop them

x["has_full_facilities"] = x["has_main_facilities"] * x["has_building_facilities"]  # combine the two features

x.drop(columns=["has_building_facilities", "has_main_facilities"], axis=1, inplace=True) # drop the old ones


x_train, x_test, y_train, y_test = split_data(x, y, test_size=0.2)

vifs_scores(x_train)

VIF for has_security_features: 61.10623358201506
VIF for has_parking_safety: 3.0624087432647027
VIF for has_other_features: 10.841117514906639
VIF for HighFloor: 1.8373621993767797
VIF for MidFloor: 2.572882351037443
VIF for has_modern: 10.995057139408047
VIF for has_facilities: 6.928464098922686
VIF for has_green_view: 5.632126392757028
VIF for has_water_feature: 13.444097944574363
VIF for Bedroom_1: 12.419923963247228
VIF for Bedroom_2: 19.514926603739983
VIF for Bedroom_3: 55.28890935297101
VIF for Bedroom_above 3: 20.029855542685503
VIF for Bathroom_1: 1.3958797100302558
VIF for Bathroom_2: 2.679557991719853
VIF for Bathroom_above 2: 4.033588929985707
VIF for city_Mansoura: 10.578552676796358
VIF for city_New Administrative Capital: 8.70763660049038
VIF for city_New Cairo: 1.8996408355341152
VIF for city_North Coast: 14.969070274513877
VIF for city_Old Cairo: 5.411669059712746
VIF for propertySubType_Attached Houses: 6.999659433616034
VIF for propertySubType_Office: 7.5810015047397

- Now we can perform the model selection method to get the best model

In [7]:
train_df = x_train.join(y_train)
logit_model, removed_vars, _ = general_to_specific(train_df, "propertyCategory_Commercial", x_train.columns, reg_type="Logistic")

         Current function value: 0.000000
         Iterations: 35
Removing Bedroom_above 3 due to corrected p=0.9713
         Current function value: 0.000002
         Iterations: 35
No variable exceeds corrected p-value or VIF threshold. Selection complete.

Final VIFs:
                         Variable      VIF
                    city_Mansoura 9.972000
                       has_modern 8.643612
                has_water_feature 7.177958
  city_New Administrative Capital 7.157430
  propertySubType_Attached Houses 6.814521
                 city_North Coast 6.177448
                         log(sqm) 5.609191
           propertySubType_Office 4.788798
                        Bedroom_1 4.380754
            has_security_features 3.470993
                   has_facilities 3.462001
                 Bathroom_above 2 3.445843
                   city_Old Cairo 3.354474
               has_other_features 3.350824
               has_parking_safety 3.192599
                   has_green_view 3.1592